## Spark write Hive table

In [1]:
import pyspark
import os
import sys
from pyspark.sql import SparkSession

spark_home = "/home/jovyan/.cache/pip/wheels/ae/78/cb/924eaddf18fb5bd07b68d76b6f706674e120883faa620a8d12"
os.environ['HADOOP_USER_NAME']="anonymous"

spark = None

try:
    spark = SparkSession.builder \
        .appName("PySpark SQL Example") \
        .config("spark.plugins", "org.apache.gravitino.spark.connector.plugin.GravitinoSparkPlugin") \
        .config("spark.jars", "/tmp/gravitino/packages/iceberg-spark-runtime-3.4_2.12-1.5.2.jar,/tmp/gravitino/packages/gravitino-spark-connector-runtime-3.4_2.12-0.9.1.jar") \
        .config("spark.sql.gravitino.uri", "http://gravitino:8090") \
        .config("spark.sql.gravitino.metalake", "metalake_demo") \
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,org.apache.paimon.spark.extensions.PaimonSparkSessionExtensions") \
        .config("spark.sql.catalog.catalog_rest", "org.apache.iceberg.spark.SparkCatalog") \
        .config("spark.sql.catalog.catalog_rest.type", "rest") \
        .config("spark.sql.catalog.catalog_rest.uri", "http://gravitino:9001/iceberg/") \
        .config("spark.sql.catalog.catalog_rest.warehouse", "hdfs://hive:9000/user/iceberg/warehouse/") \
        .config("spark.sql.catalog.catalog_hive", "org.apache.gravitino.spark.connector.hive.GravitinoHiveCatalogSpark34") \
        .config("spark.sql.catalog.catalog_hive.spark.sql.hive.metastore.jars.path", "file:///opt/spark/jars/*") \
        .config("spark.sql.catalog.paimon", "org.apache.paimon.spark.SparkCatalog") \
        .config("spark.sql.catalog.paimon.warehouse", "hdfs://hive:9000/user/paimon/warehouse/") \
        .config("spark.locality.wait.node", "0") \
        .config("spark.sql.warehouse.dir", "hdfs://hive:9000/user/hive/warehouse") \
        .enableHiveSupport() \
        .getOrCreate()
    
    print("SparkSession successfully created!")

except Exception as e:
    print("Error when create SparkSession:", e, file=sys.stderr)
    if spark is not None:
        spark.stop()
    sys.exit(1)

SparkSession successfully created!


In [2]:
spark.sql("use catalog_hive")
spark.sql("show databases").show()

+---------+
|namespace|
+---------+
|  default|
|  product|
|    sales|
+---------+



In [3]:
spark.sql("CREATE DATABASE IF NOT EXISTS product;")
spark.sql("USE product;")
spark.sql("CREATE TABLE IF NOT EXISTS employees (id INT, name STRING, age INT) PARTITIONED BY (department STRING) STORED AS PARQUET;")
spark.sql("DESC TABLE EXTENDED employees;").show()

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|                  id|                 int|   null|
|                name|              string|   null|
|                 age|                 int|   null|
|          department|              string|   null|
|# Partition Infor...|                    |       |
|          # col_name|           data_type|comment|
|          department|              string|   null|
|                    |                    |       |
|# Detailed Table ...|                    |       |
|                Name|   product.employees|       |
|                Type|             MANAGED|       |
|            Location|hdfs://hive:9000/...|       |
|               Owner|           anonymous|       |
|    Table Properties|[hive.stored-as=P...|       |
+--------------------+--------------------+-------+



In [4]:
spark.sql("INSERT OVERWRITE TABLE employees PARTITION(department='Engineering') VALUES (1, 'John Doe', 30), (2, 'Jane Smith', 28);")
spark.sql("INSERT OVERWRITE TABLE employees PARTITION(department='Marketing') VALUES (3, 'Mike Brown', 32);")
spark.sql("SELECT * from employees").show()

+---+----------+---+-----------+
| id|      name|age| department|
+---+----------+---+-----------+
|  2|Jane Smith| 28|Engineering|
|  1|  John Doe| 30|Engineering|
|  3|Mike Brown| 32|  Marketing|
+---+----------+---+-----------+



## Query the table with Trino

In [5]:
%pip install requests==2.32.3 trino==0.335.0

Note: you may need to restart the kernel to use updated packages.


In [6]:
from trino.dbapi import connect

# Create a Trino connector client
conn = connect(
    host="trino",
    port=8080,
    user="admin",
    catalog="catalog_hive",
    schema="http",
)

trino_client = conn.cursor()

In [7]:
print(trino_client.execute("SELECT * FROM catalog_hive.product.employees WHERE department = 'Engineering'").fetchall())

[[2, 'Jane Smith', 28, 'Engineering'], [1, 'John Doe', 30, 'Engineering']]


## Spark write data with Iceberg REST service

In [9]:
spark.sql("use catalog_rest;")
spark.sql("create database if not exists sales;")
spark.sql("use sales;")
spark.sql("create table if not exists customers (customer_id int, customer_name varchar(100), customer_email varchar(100));")

DataFrame[]

In [10]:
spark.sql("insert into customers (customer_id, customer_name, customer_email) values (11,'Rory Brown','rory@123.com');")
spark.sql("insert into customers (customer_id, customer_name, customer_email) values (12,'Jerry Washington','jerry@dt.com');")
spark.sql("select * from customers").show()

+-----------+----------------+--------------+
|customer_id|   customer_name|customer_email|
+-----------+----------------+--------------+
|         11|      Rory Brown|  rory@123.com|
|         12|Jerry Washington|  jerry@dt.com|
|         12|Jerry Washington|  jerry@dt.com|
|         11|      Rory Brown|  rory@123.com|
+-----------+----------------+--------------+



## Trino do federation query data with Hive and Iceberg

In [11]:
print(trino_client.execute("select * from catalog_hive.sales.customers union select * from catalog_iceberg.sales.customers").fetchall())

[[6, 'Harriet Best', 'harrietbest2890@icloud.com'], [8, 'Lenore Wilder', 'lenorewilder@aol.net'], [9, 'Raya Mcguire', 'rayamcguire@hotmail.com'], [4, 'Mia Hahn', 'miahahn@yahoo.edu'], [5, 'Quin Hurst', 'quinhurst5485@google.net'], [2, 'Perry Tyler', 'perrytyler@outlook.com'], [1, 'Nasim Duke', 'nasimduke@hotmail.net'], [7, 'Erasmus Phelps', 'erasmusphelps9105@protonmail.net'], [11, 'Rory Brown', 'rory@123.com'], [12, 'Jerry Washington', 'jerry@dt.com'], [3, 'Leah Swanson', 'leahswanson1069@protonmail.com'], [10, 'Ronan Joyner', 'ronanjoyner5549@aol.com']]


In [ ]:
import os
import signal
import ipykernel

pid = os.getpid()

print(f"Stopping Jupyter kernel {pid} ...")
os.kill(pid, signal.SIGTERM)